# NB35 — EX-2 Metrics Extraction Proof

**Objective:** Extract `get_metrics()` (~62 LOC) from `engine.py` into a standalone
`metrics.py` module. This is a read-only aggregation function over events — it has
zero interaction with SimPy yields, resources, or engine state.

## What This Notebook Proves
1. `metrics.py` has **zero SimPy imports** (HC-6)
2. `compute_metrics()` works with **mock data** — no engine needed
3. Toggle OFF (legacy) vs toggle ON (extracted) produces **identical metrics** (MC-4)
4. **Deterministic replay** — same seed = same output (HC-2)

## Strategy
- Run engine with `enable_extracted_metrics=False` → baseline metrics
- Run engine with `enable_extracted_metrics=True` → extracted metrics
- Compare all fields: total_arrivals, completed, in_system, outcomes, facilities

---
## Cell 1: Imports

In [2]:
import sys, os

# Resolve src/ whether CWD is project root (VS Code) or notebooks/phase1/ (nbconvert)
for _candidate in [
    os.path.join(os.path.abspath('.'), 'src'),
    os.path.join(os.path.abspath('..'), '..', 'src'),
    os.path.join(os.path.abspath('..'), 'src'),
]:
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

import importlib
from collections import Counter
from unittest.mock import Mock

from faer_dev.config.builder import build_engine_from_preset
from faer_dev.decisions.mode import SimulationToggles
from faer_dev.metrics import compute_metrics
print('All imports OK')

All imports OK


---
## Cell 2: Purity Check — metrics.py has zero SimPy imports (HC-6)

In [4]:
source = importlib.util.find_spec("faer_dev.metrics")
assert source is not None and source.origin is not None
with open(source.origin) as f:
    content = f.read()
assert "import simpy" not in content, "FAIL: metrics.py imports simpy"
assert "from simpy" not in content, "FAIL: metrics.py imports from simpy"
print(f'[PASS] metrics.py has zero SimPy imports (HC-6)')
print(f'       Source: {source.origin}')

[PASS] metrics.py has zero SimPy imports (HC-6)
       Source: /Users/rosstaylor/Downloads/Code Repositories/Pj FAER-Dev/Pj-FAER-Dev-Repo/Pj-FAER-Dev/notebooks/../src/faer_dev/metrics.py


---
## Cell 3: Unit Test — compute_metrics() with mock data (no engine needed)

In [6]:
# Test 1: Empty data returns zeros
transport_pool = Mock()
transport_pool.metrics.to_dict.return_value = {}
mascal_detector = Mock()
mascal_detector.activation_count = 0
mascal_detector.active = False

result = compute_metrics(
    events=[], completed_patients=[], active_patients={},
    queues={}, transport_pool=transport_pool, mascal_detector=mascal_detector,
)
assert result["total_arrivals"] == 0
assert result["completed"] == 0
assert result["in_system"] == 0
print('[PASS] Empty data returns zeros')

# Test 2: Counts ARRIVAL events correctly
events = [
    {"type": "ARRIVAL", "id": "C1"},
    {"type": "ARRIVAL", "id": "C2"},
    {"type": "TREATMENT_START", "id": "C1"},
]
result = compute_metrics(
    events=events, completed_patients=[],
    active_patients={"C1": Mock(), "C2": Mock()},
    queues={}, transport_pool=transport_pool, mascal_detector=mascal_detector,
)
assert result["total_arrivals"] == 2
assert result["in_system"] == 2
print('[PASS] Arrival count correct (2 arrivals from 3 events)')
print()
print('compute_metrics() works with mock data — no SimPy, no engine needed')

[PASS] Empty data returns zeros
[PASS] Arrival count correct (2 arrivals from 3 events)

compute_metrics() works with mock data — no SimPy, no engine needed


---
## Cell 4: Baseline — Legacy path (toggle OFF, seed=42)

In [8]:
def run_engine(toggles, seed=42, max_patients=20):
    """Run engine with COIN preset and return metrics dict."""
    engine = build_engine_from_preset("coin", seed=seed, toggles=toggles)
    return engine.run(duration=600.0, max_patients=max_patients)

baseline = run_engine(SimulationToggles(enable_extracted_metrics=False))
print('=== BASELINE (legacy metrics, toggle OFF) ===')
print(f'  total_arrivals: {baseline["total_arrivals"]}')
print(f'  completed:      {baseline["completed"]}')
print(f'  in_system:      {baseline["in_system"]}')
print(f'  outcomes:       {baseline["outcomes"]}')
print(f'  facilities:     {list(baseline["facilities"].keys())}')

=== BASELINE (legacy metrics, toggle OFF) ===
  total_arrivals: 18
  completed:      13
  in_system:      5
  outcomes:       {'RTD': 10, 'STRATEVAC': 2, 'DECEASED': 1}
  facilities:     ['R1-ALPHA', 'R2-MAIN', 'R3-THEATRE']


---
## Cell 5: Extraction Run — metrics.py path (toggle ON, seed=42)

In [10]:
extracted = run_engine(SimulationToggles(enable_extracted_metrics=True))
print('=== EXTRACTED (metrics.py, toggle ON) ===')
print(f'  total_arrivals: {extracted["total_arrivals"]}')
print(f'  completed:      {extracted["completed"]}')
print(f'  in_system:      {extracted["in_system"]}')
print(f'  outcomes:       {extracted["outcomes"]}')
print(f'  facilities:     {list(extracted["facilities"].keys())}')

=== EXTRACTED (metrics.py, toggle ON) ===
  total_arrivals: 18
  completed:      13
  in_system:      5
  outcomes:       {'RTD': 10, 'STRATEVAC': 2, 'DECEASED': 1}
  facilities:     ['R1-ALPHA', 'R2-MAIN', 'R3-THEATRE']


---
## Cell 6: Equivalence Proof — all metric fields match

In [12]:
checks = [
    ('total_arrivals', baseline['total_arrivals'] == extracted['total_arrivals']),
    ('completed',      baseline['completed'] == extracted['completed']),
    ('in_system',      baseline['in_system'] == extracted['in_system']),
    ('outcomes',       baseline['outcomes'] == extracted['outcomes']),
    ('facilities',     baseline['facilities'] == extracted['facilities']),
    ('transport',      baseline['transport'] == extracted['transport']),
    ('mascal_detector', baseline['mascal_detector'] == extracted['mascal_detector']),
]

all_pass = True
for name, ok in checks:
    status = '[PASS]' if ok else '[FAIL]'
    if not ok:
        all_pass = False
    print(f'  {status} {name}')

# Also check with seed=99
b99 = run_engine(SimulationToggles(enable_extracted_metrics=False), seed=99)
e99 = run_engine(SimulationToggles(enable_extracted_metrics=True), seed=99)
seed99_ok = (
    b99['total_arrivals'] == e99['total_arrivals']
    and b99['completed'] == e99['completed']
    and b99['outcomes'] == e99['outcomes']
)
print(f'  {"[PASS]" if seed99_ok else "[FAIL]"} seed=99 equivalence')
all_pass = all_pass and seed99_ok

print()
assert all_pass, 'EQUIVALENCE FAILED — check mismatches above'
print('EQUIVALENCE PROVEN: metrics.py extraction produces identical output')

  [PASS] total_arrivals
  [PASS] completed
  [PASS] in_system
  [PASS] outcomes
  [PASS] facilities
  [PASS] transport
  [PASS] mascal_detector
  [PASS] seed=99 equivalence

EQUIVALENCE PROVEN: metrics.py extraction produces identical output


---
## Cell 7: Determinism + Combined Toggles

In [14]:
# HC-2: Deterministic replay — two runs with same seed must match
m1 = run_engine(SimulationToggles(enable_extracted_metrics=True))
m2 = run_engine(SimulationToggles(enable_extracted_metrics=True))
assert m1 == m2
print('[PASS] Deterministic replay (HC-2): two runs with seed=42 identical')

# Combined toggles: routing + metrics both ON should still match all-OFF
m_legacy = run_engine(SimulationToggles(
    enable_extracted_routing=False, enable_extracted_metrics=False,
))
m_both = run_engine(SimulationToggles(
    enable_extracted_routing=True, enable_extracted_metrics=True,
))
assert m_legacy['total_arrivals'] == m_both['total_arrivals']
assert m_legacy['completed'] == m_both['completed']
assert m_legacy['outcomes'] == m_both['outcomes']
print('[PASS] Combined toggles (routing + metrics ON) match all-OFF')

[PASS] Deterministic replay (HC-2): two runs with seed=42 identical
[PASS] Combined toggles (routing + metrics ON) match all-OFF


---
## Summary

| Check | Status |
|-------|--------|
| `metrics.py` has zero SimPy imports (HC-6) | Verified |
| `compute_metrics()` works with mock data | Verified |
| Toggle OFF vs ON identical (seed=42) | Verified |
| Toggle OFF vs ON identical (seed=99) | Verified |
| Deterministic replay (HC-2) | Verified |
| Combined toggles (routing + metrics) | Verified |

**Conclusion:** `metrics.py` extraction is correct. Toggle can be promoted to default-ON.